# HPC Workshop — Day 1
## Fundamentals of HPC, Linux & Shell Scripting

**Topics:** HPC fundamentals, Intel HPC architecture overview, Linux basics, workspace setup, shell scripting and automation, SSH, and Git.

> Run cells sequentially. Shell commands use Bash/IPython magics and are intended for Linux or Google Colab.

# Day 1 — Fundamentals of HPC, Linux & Shell Scripting

## 1. HPC Fundamentals
HPC uses parallel processing and powerful compute resources to solve computationally intensive problems. Key concepts: clusters, nodes, CPU/core/thread, parallelism, throughput, and latency.

## 2. Intel HPC Architecture — Xeon, GPUs & AI Accelerators
Inspect CPU sockets, cores, threads, NUMA nodes, OS/kernel, and available hardware. Accelerator availability depends on the machine.

In [ ]:
import os
import subprocess

def run_command(cmd):
    result = subprocess.run(
        cmd,
        shell=True,
        capture_output=True,
        text=True
    )
    return result.stdout.strip()


cpu_model = run_command(
    "lscpu | grep 'Model name' | cut -d: -f2"
).strip()

cores = run_command("nproc")

sockets = run_command(
    "lscpu | grep 'Socket(s):' | cut -d: -f2"
).strip()

threads_per_core = run_command(
    "lscpu | grep 'Thread(s) per core:' | cut -d: -f2"
).strip()

numa_nodes = run_command(
    "lscpu | grep 'NUMA node(s):' | cut -d: -f2"
).strip()

cache_size = run_command(
    "lscpu | grep 'Cache size' | cut -d: -f2"
).strip()

vector_units = run_command(
    "lscpu | grep 'CPU(s):' | cut -d: -f2"
).strip()


print(f"CPU Model: {cpu_model}")
print(f"Cores: {cores}")
print(f"Sockets: {sockets}")
print(f"Threads per Core: {threads_per_core}")
print(f"NUMA Nodes: {numa_nodes}")
print(f"Cache Size: {cache_size}")
print(f"Vector Units: {vector_units}")

# Check for SIMD Vector Extensions (AVX-512/AVX2)

flags = run_command("grep -m 1 flags /proc/cpuinfo")

has_avx512 = "avx512" in flags
has_avx2 = "avx2" in flags
has_sse4_2 = "sse4_2" in flags

print(f"AVX512: {has_avx512}")
print(f"AVX2: {has_avx2}")
print(f"SSE4.2: {has_sse4_2}")

## 3. Linux Basics — Workspace and System Information

In [ ]:
%%bash
# create our dedicated training laboratory workspace
mkdir -p /tmp/gla_linux_lab/workspace
cd /tmp/gla_linux_lab/workspace

echo "=== Current working directory ==="
pwd

echo -e "\n== System and Kernel information ==="
uname -srm
cat /etc/os-release | grep -E '^(NAME|VERSION)='


### 3.1 Create practice directories and sample files

In [ ]:
%%bash
# create our dedicated training laboratory workspace
mkdir -p datasets logs scripts reports

touch datasets/train_sample.csv
mkdir -p datasets/test\ntouch datasets/test/sample.csv
touch scripts/run_pipeline.sh
touch config.yaml

echo "id,model_name,accuracy,loss" > datasets/results.csv
echo "1,ResNet50"

## 4. Shell Scripting & Automation

In [ ]:
import os

# 1. Create a simple executable shell script
with open('student_tutorial.sh', 'w') as f:
    f.write('''#!/bin/bash
echo "=== System Info ==="
echo "User: $(whoami)"
echo "Path: $(pwd)"
''')

os.chmod('student_tutorial.sh', 0o755)
!./student_tutorial.sh


# 2. Generate SSH Keys simply
if not os.path.exists('/root/.ssh/id_rsa'):
    !mkdir -p ~/.ssh && ssh-keygen -t rsa -b 4096 -f ~/.ssh/id_rsa -N ""


# 3. Create simple GitHub Actions template
os.makedirs('.github/workflows', exist_ok=True)

with open('.github/workflows/ci.yml', 'w') as f:
    f.write('''name: Test Script
on: [push]
''')

### 4.1 GitHub Actions workflow template

In [ ]:
import os
# 3. Create simple GitHub Actions template
os.makedirs('.github/workflows', exist_ok=True)

with open('.github/workflows/ci.yml', 'w') as f:
    f.write('''name: Test Script
on: [push]
jobs:
  run-script:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: chmod +x ./student_tutorial.sh && ./student_tutorial.sh
''')

print("\n√ Setup complete! Simple files are ready.")
print("================================================")
print("🔑 COPY YOUR PUBLIC SSH KEY BELOW:")
print("================================================")

with open('/root/.ssh/id_rsa.pub', 'r') as f:
    print(f.read().strip())

print("================================================")

## 5. Secure Shell (SSH) & Git over SSH

Replace sample identity/repository values with your own. Never share private SSH keys; share only the public `.pub` key.

In [ ]:
import os

# 1. Fill your GitHub details below:
GITHUB_USERNAME = "YOUR_GITHUB_USERNAME"
GITHUB_EMAIL = "YOUR_GITHUB_EMAIL"
REPO_NAME = "YOUR_REPOSITORY"

print("⚙️ Configuring Git details...")

os.system(f'git config --global user.name "{GITHUB_USERNAME}"')
os.system(f'git config --global user.email "{GITHUB_EMAIL}"')

print("✓ Git credentials configured locally!")

print("\n📁 Initializing Git Repository locally...")
os.system('git init')
os.system('git branch -M main')

print("✓ Git Repository initialized.")

# Adding code to remote
os.system(
    f'git remote add origin '
    f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'
)

# Add files to staging
os.system('git add .')

# Commit changes
os.system('git commit -m "Initial commit"')

# Push code to GitHub
os.system('git push -u origin main')

In [ ]:
import os

GITHUB_USERNAME = "YOUR_GITHUB_USERNAME"
REPO_NAME = "YOUR_REPOSITORY"

# 1. Add GitHub to known hosts to bypass interactive prompts
!mkdir -p ~/.ssh
!ssh-keyscan -t rsa github.com >> ~/.ssh/known_hosts

# 2. Switch the remote URL from HTTPS to SSH
!git remote set-url origin git@github.com:{GITHUB_USERNAME}/{REPO_NAME}.git

# 3. Push your code to GitHub
!git push -u origin main

In [ ]:
!ssh -T git@github.com